# 03 · Bi-LSTM on pretrained word embeddings (PyTorch)

Sequence-model comparator from the paper (Sec. 5.3): Embedding (GloVe / Word2Vec) → Bi-LSTM → concat final hidden states → dropout → softmax over 8 classes.

* Uses `text_tf` (stop-words and negations **kept** — order/negation is what the Bi-LSTM is meant to exploit).
* Hyper-parameters tuned on validation macro-F1; final model trained over several seeds → mean ± std.
* Conditions: `baseline` and `class_weight`.

**Saves:** `models/bilstm/` (weights, `vocab.json`, `config.json`), metrics/predictions in `results/`.

In [ ]:
!pip -q install torch gensim scikit-learn seaborn

In [ ]:
import os, sys, json, random, time, warnings, re, gc
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")

# ---- Project root: Google Drive folder on Colab, repo root locally ----------
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/mental-health-nlp")   # <- change if you keep the repo elsewhere
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

for d in ["data/raw", "data/interim", "data/processed", "data/splits",
          "models/classical", "models/bilstm", "models/distilbert", "models/roberta",
          "results/figures", "results/reports"]:
    (ROOT / d).mkdir(parents=True, exist_ok=True)

SEED = 42
EMOTIONS = ["Joy", "Sadness", "Anger", "Fear", "Anxiety", "Stress", "Surprise", "Neutral"]
LABEL2ID = {e: i for i, e in enumerate(EMOTIONS)}
ID2LABEL = {i: e for e, i in LABEL2ID.items()}
NEGATIVE = ["Sadness", "Anger", "Fear", "Anxiety", "Stress"]      # N in the paper (Sec. 5.6)

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass

set_seed()
print("ROOT =", ROOT)

In [ ]:
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, f1_score,
                             confusion_matrix, classification_report)

METRICS_CSV = ROOT / "results/model_metrics.csv"
LATENCY_CSV = ROOT / "results/latency_results.csv"
LABEL_IDS = list(range(len(EMOTIONS)))

def slug(condition):
    """File-system friendly condition name ('augment+class_weight' -> 'augment-class_weight')."""
    return condition.replace("+", "-")

def _upsert(path, row, keys):
    df = pd.read_csv(path) if path.exists() else pd.DataFrame()
    if len(df):
        mask = np.ones(len(df), dtype=bool)
        for k in keys:
            mask &= (df[k].astype(str) == str(row[k])).values
        df = df[~mask]
    pd.concat([df, pd.DataFrame([row])], ignore_index=True).to_csv(path, index=False)

def evaluate(y_true, y_pred, model, condition="baseline", seed=SEED, extra=None, log=True):
    """Compute all metrics, optionally upsert into results/model_metrics.csv."""
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, labels=LABEL_IDS,
                                                 average="macro", zero_division=0)
    per = f1_score(y_true, y_pred, labels=LABEL_IDS, average=None, zero_division=0)
    row = dict(model=model, condition=condition, seed=seed,
               accuracy=accuracy_score(y_true, y_pred),
               precision_macro=p, recall_macro=r, macro_f1=f, min_class_f1=per.min())
    row.update({f"f1_{e}": v for e, v in zip(EMOTIONS, per)})
    if extra:
        row.update(extra)
    if log:
        _upsert(METRICS_CSV, row, ["model", "condition", "seed"])
    return row

def log_latency(model, device, ms_per_text, note=""):
    _upsert(LATENCY_CSV, dict(model=model, device=device, ms_per_text=round(float(ms_per_text), 2), note=note),
            ["model", "device", "note"])

def plot_cm(y_true, y_pred, title, path=None, normalize=True):
    cm = confusion_matrix(y_true, y_pred, labels=LABEL_IDS)
    if normalize:
        cm = cm / cm.sum(axis=1, keepdims=True).clip(min=1)
    fig, ax = plt.subplots(figsize=(7.5, 6))
    sns.heatmap(cm, annot=True, fmt=".2f" if normalize else "d", cmap="Blues",
                xticklabels=EMOTIONS, yticklabels=EMOTIONS, ax=ax, cbar=False)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(title)
    plt.tight_layout()
    if path:
        fig.savefig(path, dpi=150)
    plt.show()

In [ ]:
SPLIT_DIR = ROOT / "data/splits"
train = pd.read_csv(SPLIT_DIR / "train.csv")
val   = pd.read_csv(SPLIT_DIR / "val.csv")
test  = pd.read_csv(SPLIT_DIR / "test.csv")
for d in (train, val, test):
    d["text_tf"] = d["text_tf"].fillna("").astype(str)
    d["text_clean"] = d["text_clean"].fillna("").astype(str)
print({k: len(v) for k, v in dict(train=train, val=val, test=test).items()})
train["label"].value_counts().reindex(EMOTIONS)

## 1 · Vocabulary & encoding (built from the training split only)

In [ ]:
import torch, torch.nn as nn, copy
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence
from collections import Counter

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

MAX_LEN, MIN_FREQ = 64, 2
TOK = re.compile(r"[a-z]+(?:'[a-z]+)?|[!?]")
def tok(t): return TOK.findall(str(t).lower().replace("\u2019", "'"))

counter = Counter(w for t in train["text_tf"] for w in tok(t))
itos = ["<pad>", "<unk>"] + [w for w, c in counter.most_common() if c >= MIN_FREQ]
stoi = {w: i for i, w in enumerate(itos)}
print("vocab size:", len(itos))

def encode(t):
    ids = [stoi.get(w, 1) for w in tok(t)][:MAX_LEN]
    return ids or [1]

class TextDS(Dataset):
    def __init__(self, df): self.x = [encode(t) for t in df["text_tf"]]; self.y = df["label_id"].values
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.x[i], self.y[i]

def collate(batch):
    xs, ys = zip(*batch)
    lens = torch.tensor([len(x) for x in xs])
    pad = torch.zeros(len(xs), int(lens.max()), dtype=torch.long)
    for i, x in enumerate(xs): pad[i, :len(x)] = torch.tensor(x)
    return pad, lens, torch.tensor(ys, dtype=torch.long)

def loader(df, bs=128, shuffle=False):
    return DataLoader(TextDS(df), batch_size=bs, shuffle=shuffle, collate_fn=collate)

dl_val, dl_test = loader(val), loader(test)

## 2 · Pretrained embeddings
`glove` = `glove-wiki-gigaword-100` (~130 MB). `word2vec` = Google News 300-d (~1.6 GB; only use with `QUICK=False`).
If the download fails (offline), the code falls back to random initialisation and says so.

In [ ]:
EMB_NAMES = {"glove": "glove-wiki-gigaword-100", "word2vec": "word2vec-google-news-300"}
_EMB_CACHE = {}

def get_embeddings(kind):
    if kind in _EMB_CACHE: return _EMB_CACHE[kind]
    try:
        import gensim.downloader as api
        kv = api.load(EMB_NAMES[kind])
        M = np.random.normal(0, 0.1, (len(itos), kv.vector_size)).astype("float32"); M[0] = 0
        hit = 0
        for w, i in stoi.items():
            if w in kv.key_to_index: M[i] = kv[w]; hit += 1
        print(f"[{kind}] pretrained vectors found for {hit}/{len(itos)} vocab words")
    except Exception as e:
        print(f"[{kind}] could not load ({type(e).__name__}) -> random init (100-d)")
        M = np.random.normal(0, 0.1, (len(itos), 100)).astype("float32"); M[0] = 0
    _EMB_CACHE[kind] = M
    return M

## 3 · Model, training loop, prediction

In [ ]:
class BiLSTMClassifier(nn.Module):
    def __init__(self, emb_matrix, hidden, n_cls, dropout):
        super().__init__()
        V, D = emb_matrix.shape
        self.emb = nn.Embedding(V, D, padding_idx=0)
        self.emb.weight.data.copy_(torch.from_numpy(emb_matrix))
        self.lstm = nn.LSTM(D, hidden, batch_first=True, bidirectional=True)
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(2 * hidden, n_cls)

    def forward(self, x, lens):
        e = self.drop(self.emb(x))
        packed = pack_padded_sequence(e, lens.cpu(), batch_first=True, enforce_sorted=False)
        _, (h, _) = self.lstm(packed)
        h = torch.cat([h[-2], h[-1]], dim=1)          # last forward + last backward hidden states
        return self.fc(self.drop(h))

N = len(train); n_c = train["label"].value_counts().reindex(EMOTIONS).values
CLASS_W = torch.tensor(N / (len(EMOTIONS) * n_c), dtype=torch.float)    # w_c = N / (|E| n_c)

@torch.no_grad()
def predict_proba(model, dl):
    model.eval(); out = []
    for x, l, _ in dl:
        out.append(torch.softmax(model(x.to(DEVICE), l), -1).cpu().numpy())
    return np.vstack(out)

def train_bilstm(seed, hidden=128, dropout=0.5, lr=1e-3, emb="glove", class_weight=False,
                 max_epochs=15, patience=3, verbose=True):
    set_seed(seed)
    model = BiLSTMClassifier(get_embeddings(emb), hidden, len(EMOTIONS), dropout).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    lossf = nn.CrossEntropyLoss(weight=CLASS_W.to(DEVICE) if class_weight else None)
    dl_tr = loader(train, shuffle=True)
    best_f1, best_state, bad = -1, None, 0
    for ep in range(1, max_epochs + 1):
        model.train(); tot = 0
        for x, l, y in dl_tr:
            opt.zero_grad()
            loss = lossf(model(x.to(DEVICE), l), y.to(DEVICE))
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
            tot += loss.item() * len(y)
        f1 = f1_score(val["label_id"], predict_proba(model, dl_val).argmax(1), average="macro")
        if verbose: print(f"  ep{ep:02d} loss={tot/len(train):.4f} val macro-F1={f1:.4f}")
        if f1 > best_f1: best_f1, best_state, bad = f1, copy.deepcopy(model.state_dict()), 0
        else:
            bad += 1
            if bad >= patience: break
    model.load_state_dict(best_state)
    return model, best_f1

## 4 · Hyper-parameter search (validation macro-F1)

In [ ]:
import itertools
QUICK = True      # True: 4 configs, GloVe only, short training.  False: full Table-4 grid (needs word2vec download).

if QUICK:
    grid = [dict(hidden=h, lr=lr, dropout=0.5, emb="glove") for h, lr in itertools.product([128, 256], [1e-3, 5e-4])]
else:
    grid = [dict(hidden=h, lr=lr, dropout=d, emb=e) for h, lr, d, e in
            itertools.product([128, 256], [1e-3, 5e-4], [0.3, 0.5], ["glove", "word2vec"])]

search = []
for cfg in grid:
    _, f1 = train_bilstm(SEED, max_epochs=8 if QUICK else 15, verbose=False, **cfg)
    search.append({**cfg, "val_macro_f1": f1}); print(cfg, f"-> {f1:.4f}")
search = pd.DataFrame(search).sort_values("val_macro_f1", ascending=False)
BEST_CFG = {k: search.iloc[0][k] for k in ["hidden", "lr", "dropout", "emb"]}
BEST_CFG["hidden"] = int(BEST_CFG["hidden"])
print("\nBest config:", BEST_CFG)
display(search)

## 5 · Final training — several seeds, two conditions
Mean ± std over seeds lets us tell real improvements from run-to-run noise (paper Sec. 5.5).

In [ ]:
SEEDS = [42, 43, 44]
RESULTS = {}
for cond, cw in [("baseline", False), ("class_weight", True)]:
    best_overall = (-1, None, None)
    for s in SEEDS:
        print(f"\n=== {cond} | seed {s} ===")
        model, vf1 = train_bilstm(s, class_weight=cw, **BEST_CFG)
        probs = predict_proba(model, dl_test); pred = probs.argmax(1)
        row = evaluate(test["label_id"].values, pred, model="bilstm", condition=cond, seed=s,
                       extra=dict(val_macro_f1=vf1))
        print(f"test acc={row['accuracy']:.4f} macro-F1={row['macro_f1']:.4f} min-class-F1={row['min_class_f1']:.4f}")
        if vf1 > best_overall[0]: best_overall = (vf1, copy.deepcopy(model.state_dict()), (pred, probs))
    # persist the best-validation seed for this condition
    torch.save(best_overall[1], ROOT / f"models/bilstm/bilstm_{cond}.pt")
    pred, probs = best_overall[2]
    pd.DataFrame({"id": test["id"], "source": test["source"], "text": test["text"],
                  "true": test["label"], "pred": [ID2LABEL[i] for i in pred], "conf": probs.max(1)}
                 ).to_csv(ROOT / f"results/predictions_bilstm_{cond}.csv", index=False)
    RESULTS[cond] = best_overall

json.dump(itos, open(ROOT / "models/bilstm/vocab.json", "w"))
json.dump({**BEST_CFG, "max_len": MAX_LEN, "emb_dim": int(get_embeddings(BEST_CFG["emb"]).shape[1]),
           "labels": EMOTIONS, "tokenizer_regex": TOK.pattern}, open(ROOT / "models/bilstm/config.json", "w"), indent=2)

In [ ]:
m = pd.read_csv(METRICS_CSV); b = m[m.model == "bilstm"]
summary = b.groupby("condition").agg(acc=("accuracy", "mean"), macro_f1_mean=("macro_f1", "mean"),
                                     macro_f1_std=("macro_f1", "std"), min_class_f1=("min_class_f1", "mean"),
                                     n_seeds=("seed", "nunique")).round(4)
display(summary)
display(b.groupby("condition")[[f"f1_{e}" for e in EMOTIONS]].mean().rename(columns=lambda c: c[3:]).round(3))

In [ ]:
pred = RESULTS["baseline"][2][0]
print(classification_report(test["label_id"], pred, target_names=EMOTIONS, digits=3, zero_division=0))
plot_cm(test["label_id"].values, pred, "Confusion matrix — Bi-LSTM (baseline, best seed)",
        path=ROOT / "results/figures/03_cm_bilstm_baseline.png")

# latency (batch=1, CPU and GPU if available)
model = BiLSTMClassifier(get_embeddings(BEST_CFG["emb"]), BEST_CFG["hidden"], 8, BEST_CFG["dropout"])
model.load_state_dict(torch.load(ROOT / "models/bilstm/bilstm_baseline.pt", map_location="cpu")); model.eval()
x, l, _ = collate([(encode(test["text_tf"].iloc[0]), 0)])
for dev in ["cpu"] + (["cuda"] if torch.cuda.is_available() else []):
    mm = model.to(dev)
    with torch.no_grad():
        for _ in range(10): mm(x.to(dev), l)
        t0 = time.perf_counter()
        for _ in range(200): mm(x.to(dev), l)
    ms = 1000 * (time.perf_counter() - t0) / 200
    log_latency("bilstm", "gpu" if dev == "cuda" else "cpu", ms, note="inference only (tokenised input), batch=1")
    print(dev, f"{ms:.2f} ms/text")

### ✅ Done
Artifacts: `models/bilstm/{bilstm_baseline.pt, bilstm_class_weight.pt, vocab.json, config.json}`.
Next: **`04_transformers.ipynb`** (needs a GPU runtime on Colab: *Runtime → Change runtime type → T4 GPU*).